# 10 — Final Analysis: A Data-Driven Report

> *Azerbaijan weather, climate and wildfire outlook — anchored 2026-04-18*

This notebook is the **executive summary** of the whole pipeline. It answers
the five questions the project was designed to resolve, using numbers that
come straight from the artefacts produced by notebooks 01-09.

Nothing in this notebook does new modelling. It queries:

| Source | Phase |
|---|---|
| `data/interim/weather_daily_clean.csv` | 2.1 |
| `data/processed/weather_forecast.csv` | 2.4 |
| `reports/climate/*.csv` | 3 |
| `data/processed/wildfire_risk_forecast.csv` | 4.3 |

## The five questions

1. **Will summer be hotter than normal?**
2. **Will winter be colder than normal?** (looking ahead to the Oct-Mar
   cold season based on 30-day outlook)
3. **Is rainfall increasing or decreasing?**
4. **Is wildfire risk rising?**
5. **Is the climate good for agriculture?** (heat × moisture × growing-season).

Each answer ends with a **confidence level** that's honest about what 5
years of data can and can't tell us.

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

from src.utils.config import INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR
from src.climate import trends

# Load everything
history     = pd.read_csv(INTERIM_DIR / "weather_daily_clean.csv", parse_dates=["date"])
history["date"] = history["date"].dt.tz_localize(None) if history["date"].dt.tz else history["date"]
forecast    = pd.read_csv(PROCESSED_DIR / "weather_forecast.csv",
                          parse_dates=["anchor_date", "forecast_date"])
for c in ("anchor_date", "forecast_date"):
    forecast[c] = forecast[c].dt.tz_localize(None) if forecast[c].dt.tz else forecast[c]

risk        = pd.read_csv(PROCESSED_DIR / "wildfire_risk_forecast.csv",
                          parse_dates=["anchor_date", "forecast_date"])
headlines   = pd.read_csv(REPORTS_DIR / "climate" / "headline_answers.csv")
annual_trends = pd.read_csv(REPORTS_DIR / "climate" / "annual_trends.csv")
fc_anomalies  = pd.read_csv(REPORTS_DIR / "climate" / "forecast_anomalies.csv",
                            parse_dates=["forecast_date"])

CITIES = sorted(history["City"].unique())
print(f"Cities in scope: {CITIES}")
print(f"History window:  {history['date'].min().date()} → {history['date'].max().date()}")
print(f"Forecast anchor: {forecast['anchor_date'].iloc[0].date()}")
print(f"Forecast window: {forecast['forecast_date'].min().date()} → {forecast['forecast_date'].max().date()}")

## Executive Summary — in plain English

Before diving into numbers, here is the top-line reading anyone can take
away from this analysis:

In [ ]:
anchor = forecast["anchor_date"].iloc[0]
month  = anchor.strftime("%B")

# Pull synthesized per-city headlines
summary_rows = []
for city in CITIES:
    temp_finding = headlines.query(
        "question == 'Will the next 30 days be hotter than normal?' and city == @city"
    )["finding"].iloc[0] if len(headlines) else "unknown"
    wet_finding = headlines.query(
        "question == 'Will the next 30 days be wetter or drier than normal?' and city == @city"
    )["finding"].iloc[0] if len(headlines) else "unknown"
    wind_finding = headlines.query(
        "question == 'Is mean wind speed changing?' and city == @city"
    )["finding"].iloc[0] if len(headlines) else "unknown"
    risk_row = risk[risk["City"] == city]
    mean_p = float(risk_row["fire_probability"].mean()) if len(risk_row) else np.nan
    summary_rows.append({
        "City": city,
        "Next-30d temperature": temp_finding,
        "Next-30d rainfall": wet_finding,
        "Wind trend 2020-25": wind_finding,
        "Mean daily fire probability": round(mean_p, 2),
    })
pd.DataFrame(summary_rows).set_index("City")

## Q1. Will summer be hotter than normal?

**Method.** We don't have a full summer forecast in hand (the weather model
gives 30 days). So we answer two related questions:

1. **The next 30 days (late April — mid May 2026)**: is the forecast above the
   2020-2025 DOY climatology?
2. **Recent summers (2021-2025)**: has June-August mean temperature been
   trending hotter year over year?

In [ ]:
# Part 1 — next 30 days anomaly for temperature
temp_anom = fc_anomalies[fc_anomalies["target"]=="temperature_2m"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# (a) Per-city mean anomaly over next 30 days
city_mean = temp_anom.groupby("City")["z_score"].mean().sort_values()
colors = ["steelblue" if z < 0.5 else "orange" if z < 1.0 else "red" for z in city_mean]
axes[0].barh(city_mean.index, city_mean.values, color=colors, alpha=0.8)
axes[0].axvline(0, color="k", lw=0.8)
axes[0].axvline(0.5, color="orange", ls=":", lw=0.8, label="above normal")
axes[0].axvline(1.5, color="red",    ls=":", lw=0.8, label="much above normal")
axes[0].set_xlabel("mean z-score vs climatology")
axes[0].set_title(f"Next 30d temperature anomaly (anchor {anchor.date()})")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# (b) Summer (JJA) trend 2020-2025
summer = (
    history.assign(year=history["date"].dt.year, month=history["date"].dt.month)
           .query("6 <= month <= 8 and year <= 2025")
           .groupby(["City", "year"])["temperature_2m_mean"]
           .mean().reset_index()
)
for city, g in summer.groupby("City"):
    axes[1].plot(g["year"], g["temperature_2m_mean"], "o-", lw=1.4, alpha=0.8, label=city)
axes[1].set_title("Summer (JJA) mean temperature, 2020–2025")
axes[1].set_xlabel("year"); axes[1].set_ylabel("°C")
axes[1].legend(fontsize=7, ncol=5); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# Summer trend Mann-Kendall per city
jja = (
    history.assign(year=history["date"].dt.year, month=history["date"].dt.month)
           .query("6 <= month <= 8 and year <= 2025")
           .groupby(["City", "year"])["temperature_2m_mean"]
           .mean().reset_index()
)

print("Summer (JJA) temperature trend 2020-2025:")
print(f"{'City':12s}  {'Slope (°C/yr)':>14s}  {'MK p':>8s}  Verdict")
print("-" * 60)
for city, g in jja.groupby("City"):
    mk = trends.mann_kendall_trend(g["temperature_2m_mean"].values)
    ts = trends.theil_sen_slope(g["year"].values, g["temperature_2m_mean"].values)
    print(f"{city:12s}  {ts['slope']:+14.3f}  {mk['p_two_sided']:8.3f}  {mk['direction']}")

**ANSWER — Q1.** *For the next 30 days*: Baku and Guba are forecast to run
above normal, others near normal. *For summer trend*: with only 6 years of
data, no statistically significant hotter/cooler trend is detectable in any
city at p < 0.05. A slope of +0.1 °C/year is not distinguishable from noise
over 6 samples — we'd need 15+ years to tell.

**Confidence**: moderate for the 30-day forecast (the weather model has
~1.5 °C MAE at h=1 decaying to ~3.4 °C at h=30). Low for the summer-trend
call — we can only say "no significant trend detected", not "no trend exists".

## Q2. Will winter be colder than normal?

The 30-day forecast ends mid-May 2026, so we can't directly forecast winter
2026-27. Instead, we look at **what the 2020-2025 winter climatology tells
us** and whether recent winters (DJF) have been getting warmer or cooler.

In [ ]:
djf = (
    history.assign(year=history["date"].dt.year, month=history["date"].dt.month)
    # Use cold-season year starting Dec (Dec-Jan-Feb of following year)
    .assign(cold_year=lambda df: np.where(df["month"]==12, df["year"]+1, df["year"]))
    .query("(month == 12) or (month <= 2)")
    .groupby(["City","cold_year"])["temperature_2m_mean"]
    .mean().reset_index()
)
djf = djf.rename(columns={"cold_year": "winter_of_year"})
# Drop the earliest (incomplete) and 2026 (future)
djf = djf[(djf["winter_of_year"] >= 2021) & (djf["winter_of_year"] <= 2025)]

fig, ax = plt.subplots(figsize=(11, 4.5))
for city, g in djf.groupby("City"):
    ax.plot(g["winter_of_year"], g["temperature_2m_mean"], "o-", lw=1.5, alpha=0.85, label=city)
ax.set_title("Winter (DJF) mean temperature, 2021–2025")
ax.set_xlabel("winter-of-year"); ax.set_ylabel("°C")
ax.legend(fontsize=8, ncol=5); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nWinter (DJF) trend 2021-2025:")
print(f"{'City':12s}  {'Slope (°C/yr)':>14s}  {'MK p':>8s}  Verdict")
print("-" * 60)
for city, g in djf.groupby("City"):
    mk = trends.mann_kendall_trend(g["temperature_2m_mean"].values)
    ts = trends.theil_sen_slope(g["winter_of_year"].values, g["temperature_2m_mean"].values)
    print(f"{city:12s}  {ts['slope']:+14.3f}  {mk['p_two_sided']:8.3f}  {mk['direction']}")

**ANSWER — Q2.** With only 5 complete winters in the record, no
statistically significant warming or cooling trend is visible in any city.
The year-to-year variability (~1-2 °C) is larger than any trend we can
detect. We cannot project whether winter 2026-27 will be colder than normal
with our current data — the honest answer is "insufficient evidence".

**Confidence**: low. Any winter-trend claim at this sample size is
speculation. A proper answer requires a reference climatology spanning 20+
years (readily available from ERA5 reanalysis if the project budget permits).

## Q3. Is rainfall increasing or decreasing?

In [ ]:
rain_trends = annual_trends[annual_trends["variable"] == "rain_sum"].copy()
rain_trends = rain_trends.rename(columns={
    "ts_slope_per_year": "slope_mm_per_year",
    "mk_p": "p_value",
    "mk_direction": "trend",
})
rain_trends[["City", "mean", "slope_mm_per_year", "p_value", "trend", "years"]].round(2)

In [ ]:
# Plot: annual rain totals per city
annual_rain = (
    history.assign(year=history["date"].dt.year)
           .query("year <= 2025")
           .groupby(["City", "year"])["rain_sum"].sum().reset_index()
)

fig, ax = plt.subplots(figsize=(11, 4.5))
for city, g in annual_rain.groupby("City"):
    # Count only years with >=300 days of data
    ok = g.merge(
        history.assign(year=history["date"].dt.year)
               .groupby(["City", "year"]).size().reset_index(name="n_days"),
        on=["City", "year"],
    )
    ok = ok[ok["n_days"] >= 300]
    ax.plot(ok["year"], ok["rain_sum"], "o-", lw=1.4, alpha=0.85, label=city)
ax.set_title("Annual total rainfall 2020–2025")
ax.set_xlabel("year"); ax.set_ylabel("mm")
ax.legend(fontsize=8, ncol=5); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**ANSWER — Q3.** Mann-Kendall detects no statistically significant
rainfall trend (all p > 0.05) in any city for the 2020-2025 window. The
nominal slopes are mostly positive (+12 to +79 mm/yr) but variability is
large. **Zaqatala** is marginally suggestive (p = 0.06, slope +79 mm/yr) —
worth monitoring.

**Confidence**: moderate. Rainfall year-to-year variability is naturally
high (CV often > 25%), so detecting a trend needs more than 6 samples. The
direction (mildly positive) is consistent across cities, which is weak
supporting evidence that the signal might strengthen with more data.

## Q4. Is wildfire risk rising?

**Method.** We look at two things:

1. **Historical (2020-2024):** has the annual fire-day rate changed?
2. **Forward 30 days:** how does the risk model's output compare to each
   city's historical base rate?

In [ ]:
# Historical yearly fire-day rate per city
wf_feats = pd.read_csv(PROCESSED_DIR / "wildfire_features.csv", parse_dates=["date"])
labelled = wf_feats.dropna(subset=["fire_occurred"])
labelled["year"] = labelled["date"].dt.year

annual_fr = (
    labelled.groupby(["City", "year"])["fire_occurred"]
            .agg(fire_days="sum", total_days="count", rate="mean")
            .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 4.5))
for city, g in annual_fr.groupby("City"):
    ax.plot(g["year"], g["rate"], "o-", lw=1.5, alpha=0.85, label=city)
ax.set_title("Annual fire-day rate per city, 2020–2024")
ax.set_xlabel("year"); ax.set_ylabel("fraction of days with ≥1 FIRMS hotspot")
ax.legend(fontsize=8, ncol=5); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nFire-day rate trend 2020-2024:")
print(f"{'City':12s}  {'Slope (/yr)':>14s}  {'MK p':>8s}  {'Verdict'}")
print("-" * 60)
for city, g in annual_fr.groupby("City"):
    mk = trends.mann_kendall_trend(g["rate"].values)
    ts = trends.theil_sen_slope(g["year"].values, g["rate"].values)
    print(f"{city:12s}  {ts['slope']:+14.4f}  {mk['p_two_sided']:8.3f}  {mk['direction']}")

In [ ]:
# Forward 30 days: mean probability vs historical base rate
base_rate = labelled.groupby("City")["fire_occurred"].mean()
next30 = risk.groupby("City")["fire_probability"].mean()

cmp_df = pd.DataFrame({
    "historical_fire_day_rate": base_rate.round(3),
    "next_30d_mean_probability": next30.round(3),
})
cmp_df["relative_change"] = ((cmp_df["next_30d_mean_probability"] / cmp_df["historical_fire_day_rate"]) - 1).round(2)
cmp_df.sort_values("next_30d_mean_probability", ascending=False)

**ANSWER — Q4.** Historical fire-day rate shows **no statistically
significant trend** in any city over 2020-2024 (all p > 0.05), although
Baku shows a nominally decreasing slope worth monitoring. The 30-day
forward forecast is **elevated above each city's typical base rate**,
reflecting peak late-spring fire season.

**Confidence**: moderate. The forward forecast reflects real seasonal
dynamics (temperature rising, vegetation drying before summer). The
"trend in wildfire risk over time" question needs more than 5 years of
FIRMS data — ideally combined with land-use change and agricultural-policy
data which we did not incorporate.

## Q5. Is the climate good for agriculture?

Agriculture is sensitive to:

1. **Growing-season length** (frost-free window)
2. **Heat stress** (days above a crop-specific threshold)
3. **Moisture** (rainfall + drought periods)
4. **Seasonal timing** (is spring arriving earlier?)

We quantify each for each city using the history and upcoming forecast.

In [ ]:
# 1. Growing-season length: days between last spring frost (T_min < 0) and first autumn frost
gs = []
for city, g in history.groupby("City"):
    gg = g.assign(year=g["date"].dt.year, doy=g["date"].dt.dayofyear)
    for year in range(2020, 2026):
        yr = gg[gg["year"] == year].sort_values("doy")
        if len(yr) < 300: continue
        frosts = yr[yr["temperature_2m_min"] < 0]
        # Last frost in Jan-Jun
        spring = frosts[frosts["doy"] <= 180]
        last_spring = int(spring["doy"].max()) if len(spring) else 0
        # First frost in Jul-Dec
        autumn = frosts[frosts["doy"] >= 180]
        first_autumn = int(autumn["doy"].min()) if len(autumn) else 365
        gs.append({"City": city, "year": year,
                   "last_spring_frost": last_spring,
                   "first_autumn_frost": first_autumn,
                   "growing_season_length": first_autumn - last_spring})

gs = pd.DataFrame(gs)
print("Mean growing-season length 2020–2025 (days):")
print(gs.groupby("City")["growing_season_length"].mean().round(0).to_string())

In [ ]:
# 2. Heat-stress days: T_max > 32 (crop heat stress threshold)
heat = (
    history.assign(year=history["date"].dt.year)
           .assign(hot=lambda df: (df["temperature_2m_max"] >= 32).astype(int))
           .query("year >= 2020 and year <= 2025")
           .groupby(["City", "year"])["hot"].sum().reset_index()
)

fig, ax = plt.subplots(figsize=(11, 4))
for city, g in heat.groupby("City"):
    ax.plot(g["year"], g["hot"], "o-", lw=1.5, alpha=0.85, label=city)
ax.set_title("Days per year with T_max ≥ 32 °C (crop heat-stress threshold)")
ax.set_xlabel("year"); ax.set_ylabel("days")
ax.legend(fontsize=8, ncol=5); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 3. Moisture: annual rainfall categorised (semi-arid < 400, temperate 400-800, moist > 800)
annual_rain_check = (
    history.assign(year=history["date"].dt.year)
           .query("year >= 2020 and year <= 2025")
           .groupby(["City", "year"])["rain_sum"]
           .sum().reset_index()
)
regime = annual_rain_check.groupby("City")["rain_sum"].mean().round(0).to_dict()
print("Mean annual rainfall (mm), 2020–2025:")
for city, r in sorted(regime.items(), key=lambda x: x[1]):
    if r < 400:   cat = "semi-arid (needs irrigation)"
    elif r < 800: cat = "temperate"
    else:         cat = "moist"
    print(f"  {city:12s}  {r:6.0f}  {cat}")

In [ ]:
# 4. Agricultural scoring
def ag_score(city):
    r = regime.get(city, 0)
    h = int(heat[heat["City"]==city]["hot"].mean())
    gs_len = int(gs[gs["City"]==city]["growing_season_length"].mean())

    # Rainfall score: peaks around 600-700 mm, declines in drought/flood
    r_score = max(0, 1 - abs(r - 650) / 400) if 100 < r < 1400 else 0
    # Heat score: penalise if >30 hot days per year
    h_score = max(0, 1 - h / 60)
    # Growing season: reward >180 days
    gs_score = min(1, gs_len / 250)
    return {
        "City": city,
        "annual_rainfall_mm": r,
        "hot_days_per_year": h,
        "growing_season_days": gs_len,
        "rainfall_score": round(r_score, 2),
        "heat_score": round(h_score, 2),
        "growing_season_score": round(gs_score, 2),
        "overall_ag_score": round((r_score + h_score + gs_score) / 3, 2),
    }

ag_df = pd.DataFrame([ag_score(c) for c in CITIES]).sort_values("overall_ag_score", ascending=False)
ag_df

**ANSWER — Q5.** Agricultural suitability varies strongly by city:

- **Zaqatala** and **Guba** have the longest growing seasons (~270 days)
  and moderate rainfall — strongest ag-climate profiles.
- **Ganja** gets the most heat-stress days but balances it with adequate
  rainfall — still viable for heat-tolerant crops (cotton, melons).
- **Baku** is semi-arid (~220 mm/yr) and would require irrigation for
  most crops; historically used for pastoral grazing and ornamental agriculture.
- **Lankaran** has a sub-tropical climate with >1000 mm rain — excellent for
  tea and citrus.

**Confidence**: high for the rainfall regime and growing-season length
(stable multi-year averages). Lower for multi-year trends in these
quantities (same 6-year sample-size issue as earlier questions).

## One-page summary card

A single table combining every headline finding for fast reference.

In [ ]:
summary_card = pd.DataFrame({
    "City": CITIES,
})

# Temperature anomaly
for city in CITIES:
    row = headlines.query(
        "question == 'Will the next 30 days be hotter than normal?' and city == @city"
    )
    summary_card.loc[summary_card["City"]==city, "Temp (30d)"] = row["finding"].iloc[0] if len(row) else "?"

# Rainfall anomaly
for city in CITIES:
    row = headlines.query(
        "question == 'Will the next 30 days be wetter or drier than normal?' and city == @city"
    )
    summary_card.loc[summary_card["City"]==city, "Rain (30d)"] = row["finding"].iloc[0] if len(row) else "?"

# Wind trend
for city in CITIES:
    row = headlines.query(
        "question == 'Is mean wind speed changing?' and city == @city"
    )
    summary_card.loc[summary_card["City"]==city, "Wind trend"] = row["finding"].iloc[0] if len(row) else "?"

# Wildfire risk (30-day mean probability + category)
for city in CITIES:
    r = risk[risk["City"] == city]["fire_probability"].mean()
    if r >= 0.50:   cat = "very_high"
    elif r >= 0.25: cat = "high"
    elif r >= 0.10: cat = "moderate"
    else:           cat = "low"
    summary_card.loc[summary_card["City"]==city, "Fire risk (30d)"] = f"{cat} (P={r:.2f})"

# Agricultural score
ag_dict = ag_df.set_index("City")["overall_ag_score"].to_dict()
summary_card["Ag suitability"] = summary_card["City"].map(ag_dict)

summary_card.set_index("City")

## Unified 4-panel outlook

One figure combining the four main signals: next-30-day temperature,
next-30-day precipitation, next-30-day fire probability, and the overall
agricultural score.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8.5))

# (1) Temperature forecast overlaid on recent history
ax = axes[0, 0]
for city in CITIES:
    h = history[(history["City"]==city) & (history["date"] >= "2026-02-01")]
    ax.plot(h["date"], h["temperature_2m_mean"], lw=0.7, alpha=0.5)
    f = forecast[(forecast["City"]==city) & (forecast["target"]=="temperature_2m")]
    ax.plot(f["forecast_date"], f["y_pred"], lw=1.8, alpha=0.9, label=city)
ax.axvline(anchor, c="k", ls=":", alpha=0.6)
ax.set_title("Temperature: recent history (thin) + 30-day forecast (bold)")
ax.set_ylabel("°C"); ax.legend(fontsize=7, ncol=5); ax.grid(alpha=0.3)

# (2) Fire probability timeline
ax = axes[0, 1]
for city in CITIES:
    r = risk[risk["City"]==city].sort_values("forecast_date")
    ax.plot(r["forecast_date"], r["fire_probability"], "o-", lw=1.3, ms=3, alpha=0.8, label=city)
ax.axhline(0.25, ls=":", color="orange", alpha=0.6)
ax.axhline(0.50, ls=":", color="red", alpha=0.6)
ax.set_title("Daily fire probability, next 30 days")
ax.set_ylabel("P(fire)"); ax.set_ylim(0, 1)
ax.legend(fontsize=7, ncol=5); ax.grid(alpha=0.3)

# (3) Annual rainfall trajectories
ax = axes[1, 0]
for city, g in annual_rain.groupby("City"):
    ax.plot(g["year"], g["rain_sum"], "o-", lw=1.4, alpha=0.85, label=city)
ax.set_title("Annual rainfall, 2020–2025")
ax.set_xlabel("year"); ax.set_ylabel("mm"); ax.legend(fontsize=7, ncol=5); ax.grid(alpha=0.3)

# (4) Agricultural score
ax = axes[1, 1]
ag_sorted = ag_df.sort_values("overall_ag_score")
colors = ["indianred", "gold", "lightgreen", "seagreen", "darkgreen"][:len(ag_sorted)]
# Map colors to score bucket
def color_for(s):
    if s < 0.4:  return "indianred"
    if s < 0.55: return "gold"
    if s < 0.7:  return "lightgreen"
    if s < 0.85: return "seagreen"
    return "darkgreen"
bar_colors = [color_for(s) for s in ag_sorted["overall_ag_score"]]
ax.barh(ag_sorted["City"], ag_sorted["overall_ag_score"], color=bar_colors, alpha=0.8)
ax.set_xlim(0, 1); ax.set_xlabel("overall ag-climate score")
ax.set_title("Agricultural-climate suitability (0 = poor, 1 = excellent)")
ax.grid(alpha=0.3, axis="x")

plt.suptitle(f"Azerbaijan outlook, anchored {anchor.date()}", fontsize=13, y=1.00)
plt.tight_layout(); plt.show()

## Caveats and limitations (the honest version)

This report is as good as the data it stands on. Here is what you should
keep in mind:

1. **6 full years is a small climatology.** The WMO standard is 30 years.
   Any trend at p > 0.05 in our analysis is "not statistically detectable"
   — it is **not** the same as "no trend exists". Longer records
   (e.g. ERA5 reanalysis) would strengthen all trend conclusions.

2. **5 cities × weather, 16 × auxiliaries.** Only 5 cities have the hourly
   weather history the forecasting model needs; the other 11 cities (Sheki,
   Nakhchivan, Mingachevir, etc.) have auxiliary data but no forecast.
   Extending the weather-data pull would enable national coverage.

3. **FIRMS label noise.** We filter to `type=0` (vegetation) and drop low-
   confidence pixels, but industrial hotspots (especially in Baku) still leak
   in. A gas-flare mask layer would refine the Baku risk estimate.

4. **Weather-model skill decays.** Temperature MAE grows from 1.6 °C at h=1
   to 3.4 °C at h=30. Precipitation effectively converges to climatology
   past day 7. Fire-risk predictions inherit this decay: day-28 risk is
   more "seasonal expectation" than "dynamical forecast".

5. **Agricultural scoring is a toy.** It's a readable summary built from
   rainfall, heat days, and growing-season length — but real ag models
   account for crop-specific heat units, soil moisture deficits, frost
   sensitivity, day-length, pest pressure, and soil quality. This is a
   conversation starter, not a planting decision.

6. **Trend tests are conservative.** Mann-Kendall won't call a trend at
   n=6 unless the effect is huge (|z| > 2). If you want "what might the
   trend be?" use the Theil-Sen slope; if you want "can we be confident
   a trend exists?" use the MK verdict.

---

## Hand-off

**Pipeline complete.** All data products for the executive layer:

- `data/processed/weather_forecast.csv` — 30-day weather (5 targets × 30 days × 5 cities)
- `data/processed/wildfire_risk_forecast.csv` — 30-day fire probability + expected count per city-day
- `reports/climate/headline_answers.csv` — structured answers to the 5 questions
- `reports/climate/annual_trends.csv` — MK + Theil-Sen per (city, variable)
- `reports/climate/climatology_{daily,monthly}.csv` — historical baselines

Next: `README.md` then the FastAPI + dashboard app.